In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv("D:\Data Visualisation\git exms\Message_Intelligence_Dataset_5200 - Message_Intelligence_Dataset_5200.csv.csv")


<>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
C:\Users\Admin\AppData\Local\Temp\ipykernel_12964\3266551353.py:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  df = pd.read_csv("D:\Data Visualisation\git exms\Message_Intelligence_Dataset_5200 - Message_Intelligence_Dataset_5200.csv.csv")


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   message_id               5200 non-null   int64  
 1   message_text             5200 non-null   str    
 2   message_length           5200 non-null   int64  
 3   word_count               5200 non-null   int64  
 4   num_urls                 5200 non-null   int64  
 5   num_digits               5200 non-null   int64  
 6   num_special_chars        5200 non-null   int64  
 7   spam_keyword_score       5200 non-null   int64  
 8   legit_keyword_score      5200 non-null   int64  
 9   sender_activity_score    5094 non-null   float64
 10  sender_account_age_days  5087 non-null   float64
 11  messages_sent_last_24h   5038 non-null   float64
 12  timestamp                5200 non-null   str    
 13  hour_of_day              5200 non-null   int64  
 14  day_of_week              5200 non-n

In [5]:
# Target variable (y) and Input features (X)
y = df['spam_label']
X = df.drop(columns=['spam_label', 'message_id', 'message_text', 'timestamp'])


In [6]:
from sklearn.model_selection import train_test_split

# Split the data, using stratify to keep spam ratios equal
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [7]:
from sklearn.preprocessing import StandardScaler

# Scale the features so large numbers don't overwhelm the KNN algorithm
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Part C: Baseline Model – K-Nearest Neighbors

Question 9: Implement K-Nearest Neighbors (KNN) classifier

In [9]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer

# Initialize and train the basic KNN model
knn_basic = KNeighborsClassifier(n_neighbors=5)
# Impute missing values in the scaled feature matrices
imputer = SimpleImputer(strategy='mean')
X_train_scaled = imputer.fit_transform(X_train_scaled)
X_test_scaled = imputer.transform(X_test_scaled)

# Initialize and train the basic KNN model
knn_basic = KNeighborsClassifier(n_neighbors=5)
knn_basic.fit(X_train_scaled, y_train)

# Generate predictions and evaluate
y_pred_basic = knn_basic.predict(X_test_scaled)
print(classification_report(y_test, y_pred_basic))

# Generate predictions and evaluate
y_pred_basic = knn_basic.predict(X_test_scaled)
print(classification_report(y_test, y_pred_basic))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       845
           1       1.00      1.00      1.00       195

    accuracy                           1.00      1040
   macro avg       1.00      1.00      1.00      1040
weighted avg       1.00      1.00      1.00      1040

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       845
           1       1.00      1.00      1.00       195

    accuracy                           1.00      1040
   macro avg       1.00      1.00      1.00      1040
weighted avg       1.00      1.00      1.00      1040



Question 11: Analyze how distance metrics affect predictions

In [10]:
# Test K values to find the best fit (avoiding underfitting/overfitting)
for k in [3, 5, 7, 9, 15]:
    knn_k = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn_k.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_scaled))
    print(f"K={k} -> Accuracy: {acc:.4f}")


K=3 -> Accuracy: 1.0000
K=5 -> Accuracy: 1.0000
K=7 -> Accuracy: 1.0000
K=9 -> Accuracy: 1.0000
K=15 -> Accuracy: 0.9990


Question 12: Identify cases where KNN misclassifies messages



In [11]:
# Train the best version of the model found in Q10 and Q11
best_knn = KNeighborsClassifier(n_neighbors=5, metric='manhattan', weights='distance')
best_knn.fit(X_train_scaled, y_train)
y_pred_best = best_knn.predict(X_test_scaled)

# Find exactly where the model guessed wrong
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_best})
fp = len(results_df[(results_df['Actual'] == 0) & (results_df['Predicted'] == 1)])
fn = len(results_df[(results_df['Actual'] == 1) & (results_df['Predicted'] == 0)])

print(f"False Positives (Safe text flagged as Spam): {fp}")
print(f"False Negatives (Spam that bypassed filter): {fn}")

False Positives (Safe text flagged as Spam): 0
False Negatives (Spam that bypassed filter): 0


# Part D: Support Vector Machine Classifier

Question 13: Implement SVM with Linear and RBF kernels

In [12]:
from sklearn.svm import SVC

# Linear Kernel SVM
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)

print("--- Linear SVM Performance ---")
print(classification_report(y_test, y_pred_linear))

# RBF Kernel SVM (handles non-linear relationships better)
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = svm_rbf.predict(X_test_scaled)

print("\n--- RBF SVM Performance ---")
print(classification_report(y_test, y_pred_rbf))

--- Linear SVM Performance ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       845
           1       1.00      1.00      1.00       195

    accuracy                           1.00      1040
   macro avg       1.00      1.00      1.00      1040
weighted avg       1.00      1.00      1.00      1040


--- RBF SVM Performance ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       845
           1       1.00      1.00      1.00       195

    accuracy                           1.00      1040
   macro avg       1.00      1.00      1.00      1040
weighted avg       1.00      1.00      1.00      1040



Question 14: Analyze margin separation and support vectors

In [13]:
# Extract the support vectors (the specific messages acting as boundary markers)
print("--- Linear SVM Support Vectors ---")
print(f"Total Support Vectors: {sum(svm_linear.n_support_)}")
print(f"Per Class (Safe, Spam): {svm_linear.n_support_}")

print("\n--- RBF SVM Support Vectors ---")
print(f"Total Support Vectors: {sum(svm_rbf.n_support_)}")
print(f"Per Class (Safe, Spam): {svm_rbf.n_support_}")


--- Linear SVM Support Vectors ---
Total Support Vectors: 9
Per Class (Safe, Spam): [5 4]

--- RBF SVM Support Vectors ---
Total Support Vectors: 141
Per Class (Safe, Spam): [75 66]


In [14]:
# Compare the best KNN model with the best SVM model
acc_knn = accuracy_score(y_test, y_pred_best)
acc_svm_rbf = accuracy_score(y_test, y_pred_rbf)

print("--- Model Comparison: SVM vs KNN ---")
print(f"Best KNN Accuracy (Manhattan, K=5): {acc_knn:.4f}")
print(f"RBF SVM Accuracy:                   {acc_svm_rbf:.4f}")

--- Model Comparison: SVM vs KNN ---
Best KNN Accuracy (Manhattan, K=5): 1.0000
RBF SVM Accuracy:                   1.0000


# Part E: Naive Bayes Classifier & Probability

Question 16: Implement Naive Bayes Classifier

In [15]:
from sklearn.naive_bayes import GaussianNB

# Implement Gaussian Naive Bayes (used when features are continuous/scaled numbers)
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)
y_pred_gnb = gnb.predict(X_test_scaled)

print("--- Naive Bayes Performance ---")
print(classification_report(y_test, y_pred_gnb))

--- Naive Bayes Performance ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       845
           1       1.00      1.00      1.00       195

    accuracy                           1.00      1040
   macro avg       1.00      1.00      1.00      1040
weighted avg       1.00      1.00      1.00      1040



Question 17: Manually compute conditional probabilities

In [16]:
import numpy as np
from scipy.stats import norm

# Pick a single random message from the test set to analyze manually
sample_idx = 0
sample_features = X_test_scaled[sample_idx]

# Extract the mathematical means (theta) and variances (var) learned by the model
mean_safe, mean_spam = gnb.theta_[0], gnb.theta_[1]
var_safe, var_spam = gnb.var_[0], gnb.var_[1]

# Calculate Conditional Probabilities: P(Features | Safe) and P(Features | Spam)
# Naive Bayes assumes features are independent, so we multiply their individual probabilities together
prob_x_given_safe = np.prod(norm.pdf(sample_features, loc=mean_safe, scale=np.sqrt(var_safe)))
prob_x_given_spam = np.prod(norm.pdf(sample_features, loc=mean_spam, scale=np.sqrt(var_spam)))

print(f"P(Features | Safe) = {prob_x_given_safe}")
print(f"P(Features | Spam) = {prob_x_given_spam}")


P(Features | Safe) = 1928.2661218529129
P(Features | Spam) = 0.0


Question 19: Compare theoretical calculations with model predictions

In [18]:
# Ask the actual scikit-learn model to predict the probabilities for the exact same message
model_probs = gnb.predict_proba([sample_features])[0]

print("--- Comparison ---")
print(f"Theoretical (Manual) Calculation : Safe = {prob_x_given_safe:.6f}, Spam = {prob_x_given_spam:.6f}")
print(f"Model (gnb.predict_proba) Output : Safe = {model_probs[0]:.6f}, Spam = {model_probs[1]:.6f}")


--- Comparison ---
Theoretical (Manual) Calculation : Safe = 1928.266122, Spam = 0.000000
Model (gnb.predict_proba) Output : Safe = 1.000000, Spam = 0.000000


# Part F: Model Comparison & Evaluation

Question 20: Evaluate all models using appropriate classification metrics

In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Dictionary of all our model predictions from previous steps
model_predictions = {
    "KNN (Manhattan, K=5)": y_pred_best,
    "SVM (Linear)": y_pred_linear,
    "SVM (RBF)": y_pred_rbf,
    "Naive Bayes (Gaussian)": y_pred_gnb
}

# Calculate Accuracy, Precision, Recall, and F1 for each model
results = []
for model_name, y_pred in model_predictions.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })

# Display as a clean comparison table
comparison_df = pd.DataFrame(results).round(4)
print("--- Q20: Final Model Evaluation Metrics ---")
print(comparison_df.to_string(index=False))

--- Q20: Final Model Evaluation Metrics ---
                 Model  Accuracy  Precision  Recall  F1 Score
  KNN (Manhattan, K=5)       1.0        1.0     1.0       1.0
          SVM (Linear)       1.0        1.0     1.0       1.0
             SVM (RBF)       1.0        1.0     1.0       1.0
Naive Bayes (Gaussian)       1.0        1.0     1.0       1.0
